<a href="https://colab.research.google.com/github/khr26/VU-thesis-krao/blob/main/01_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/ORD_replication'
os.makedirs(DRIVE_PATH, exist_ok=True)
print('drive ready')

Mounted at /content/drive
drive ready


In [2]:
#2
import os
if not os.path.exists(f'{DRIVE_PATH}/ORD/.git'):
    !git clone https://github.com/Annie2603/ORD.git {DRIVE_PATH}/ORD
else:
    print('repo already present')
os.chdir(f'{DRIVE_PATH}/ORD')
print('cwd:', os.getcwd())

fatal: destination path '/content/drive/MyDrive/ORD_replication/ORD' already exists and is not an empty directory.
cwd: /content/drive/MyDrive/ORD_replication/ORD


In [3]:
#3
!pip install -q pandas numpy matplotlib scikit-learn scipy

In [4]:
#4
import os
if not os.path.exists('adult.data'):
    !wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -O adult.data
    !wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test -O adult.test
else:
    print('raw files already downloaded')

raw files already downloaded


In [5]:
#5
import pandas as pd, os

RAW_COLS = ['age','workclass','fnlwgt','education','education-num','marital-status',
            'occupation','relationship','race','sex','capital-gain','capital-loss',
            'hours-per-week','native-country','income']

train_raw = pd.read_csv('adult.data', header=None, names=RAW_COLS, sep=', ', engine='python')
test_raw  = pd.read_csv('adult.test', header=None, names=RAW_COLS, sep=', ', engine='python', skiprows=1)
test_raw['income'] = test_raw['income'].str.rstrip('.')   # test labels have a trailing dot

df = pd.concat([train_raw, test_raw], ignore_index=True)
df = df[~(df == '?').any(axis=1)].reset_index(drop=True)             # drop rows with missing values
df = df.drop(columns=['education']).rename(                          # keep numeric education-num
    columns={'sex':'gender', 'education-num':'educational-num'})
df['income'] = df['income'].map({'<=50K':0, '>50K':1})              # minority (>50K) = 1

cols = ['age','workclass','fnlwgt','educational-num','marital-status','occupation',
        'relationship','race','gender','capital-gain','capital-loss','hours-per-week',
        'native-country','income']
df = df[cols]

os.makedirs('data/adult', exist_ok=True)
df.to_csv('data/adult/original.csv', index=False)
print('original.csv:', df.shape, '| minority:', f"{(df['income']==1).mean():.2%}")
print(df['income'].value_counts().sort_index().to_dict())

original.csv: (45222, 14) | minority: 24.78%
{0: 34014, 1: 11208}


In [6]:
#6
import os, subprocess
if not (os.path.exists('data/adult/test.csv') and os.path.exists('data/adult/imbalanced_noord.csv')):
    subprocess.run(['python','preprocess.py','--dataname','adult',
                    '--testsize','4000','--imbalance_ratio','0.02','--target','income'])
else:
    print('split already built, skipping')

split already built, skipping


In [7]:
#7
import pandas as pd
train = pd.read_csv('data/adult/imbalanced_noord.csv')
test  = pd.read_csv('data/adult/test.csv')

print('imbalanced_noord:', train.shape, train['income'].value_counts().sort_index().to_dict())
print('test:', test.shape, test['income'].value_counts().sort_index().to_dict())
print('training minority:', f"{(train['income']==1).mean():.2%}")

assert set(train.columns) == set(test.columns)
assert set(train['income'].unique()) <= {0, 1}

imbalanced_noord: (32654, 14) {0: 32014, 1: 640}
test: (4000, 14) {0: 2000, 1: 2000}
training minority: 1.96%


In [8]:
#8
print('same columns as original:', set(df.columns) == set(train.columns))

same columns as original: True


In [9]:
#9
with open('detect_overlap.py') as f:
    src = f.read()
src = src.replace('RandomForestClassifier()', 'RandomForestClassifier(random_state=42)')
with open('detect_overlap.py', 'w') as f:
    f.write(src)
print('patched random_state into detect_overlap.py')

patched random_state into detect_overlap.py


In [10]:
#10
import os, subprocess
# 0 = clear majority, 1 = overlap majority, 2 = minority
# threshold 0.165 gives an overlap count close to the minority count (the paper's rule)
if not os.path.exists('data/adult/imbalanced_ord.csv'):
    subprocess.run(['python','detect_overlap.py','--dataname','adult',
                    '--target','income','--threshold','0.165'])
else:
    print('imbalanced_ord.csv already built, skipping')

imbalanced_ord.csv already built, skipping


In [11]:
#11
import pandas as pd
ord_train = pd.read_csv('data/adult/imbalanced_ord.csv')

counts = ord_train['cond'].value_counts().sort_index().to_dict()
print('imbalanced_ord:', ord_train.shape)
print('cond counts:', counts, '(0=clear majority, 1=overlap majority, 2=minority)')

n_overlap  = (ord_train['cond'] == 1).sum()
n_majority = (ord_train['cond'] != 2).sum()
print('overlap:', n_overlap, '| as % of majority:', f'{n_overlap/n_majority:.2%}')

imbalanced_ord: (32654, 14)
cond counts: {0: 31409, 1: 605, 2: 640} (0=clear majority, 1=overlap majority, 2=minority)
overlap: 605 | as % of majority: 1.89%
